In [1]:
import os, sys
sys.path.append("E:\\Dai hoc\\2526I\\dacn\\flow-matching\\pep2prob")
# os.environ["PYTHONPATH"] = "E:\\Dai hoc\\2526I\\dacn\\flow-matching\\pep2prob"

In [ ]:
import torch
from torch.nn.functional import mse_loss
import esm
from flow_matching.path import CondOTProbPath
# from flow_matching.loss import 

In [3]:
from my_model.mlp.mlp import FlowMLP
from p2p_dataset import Pep2ProbDataset


In [ ]:
peptide_embedding_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
peptide_embedding_model.eval()
batch_converter = alphabet.get_batch_converter()
pep2prob_ds = Pep2ProbDataset(data_dir="data")

In [ ]:
def extract_ds_row(ds_row: tuple):
    (precursor, probs, prob_masks, info) = ds_row
    return {
        "precursor": precursor,
        "probs": probs,
        "prob_masks": prob_masks,
        "info": info
    }

def get_peptide_embs(peptides: list[str]):
    batch_labels, batch_strs, batch_tokens = batch_converter(
        [(f"peptide_{i}", pep) for i, pep in enumerate(peptides)]
    )
    with torch.no_grad():
        results = peptide_embedding_model(batch_tokens, repr_layers=[12], return_contacts=False)
    token_representations = results["representations"][12]
    # Generate per-peptide embeddings via averaging
    peptide_embeddings = []
    for i, label in enumerate(batch_labels):
        # Remove <cls> and <eos> tokens
        embedding = token_representations[i, 1 : len(label) + 1].mean(0)
        peptide_embeddings.append(embedding)
    peptide_embeddings = torch.stack(peptide_embeddings, dim=0)
    return peptide_embeddings

def collate_fn(batch):
    peptides = [extract_ds_row(row)["precursor"] for row in batch]
    peptide_embs = get_peptide_embs(peptides)
    probs = torch.stack([extract_ds_row(row)["probs"] for row in batch], dim=0)
    prob_masks = torch.stack([extract_ds_row(row)["prob_masks"] for row in batch], dim=0)
    infos = [extract_ds_row(row)["info"] for row in batch]
    return {
        "peptide_embs": peptide_embs,
        "probs": probs,
        "prob_masks": prob_masks,
        "infos": infos
    }

In [ ]:
pep2prob_ds.train[0]

In [ ]:
collate_fn([pep2prob_ds.train[0]])